# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library in Python. We'll walk through loading the dataset defined by its Croissant schema, overviewing its record sets and fields (referenced by `@id`), extracting tabular data, and performing exploratory analysis and visualization.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is available
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR^2 dataset Croissant schema may define multiple record sets. We'll list them with their `@id`, then for each record set, show its fields (also by `@id`). All references below use the explicit Croissant `@id` fields as required.

In [ ]:
# List all record sets in the dataset schema
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
print('Available Record Sets by @id:')
for rs in metadata.record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")

# List fields for each record set
for rs in metadata.record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    fields = rs.get('fields', [])
    for f in fields:
        print(f"  Field @id: {f['@id']}  Name: {f.get('name','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. We'll extract every record set. For demonstration, we will use the first record set if multiple exist.

All access is by `@id` per Croissant best practices.

In [ ]:
# Extract data from each record set using its @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for record_set @id: {record_set_id}")
    recs = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows. Columns (field @id): {df.columns.tolist()}")

# For demonstration, select the first record set
main_record_set = record_set_ids[0] if record_set_ids else None
if main_record_set:
    print(f"\nFirst 5 rows of main record set (@id={main_record_set}):")
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Here we'll select numeric and categorical fields by their `@id` and perform some basic EDA: filtering, normalization, and grouping.

- **Note:** Substitute the `<numeric_field_id>` and `<group_field_id>` below with actual field `@id`s present in your data, as shown in the previous outputs.

In [ ]:
# EDA: Filtering, normalization, grouping by field @id

# 1. Identify a numeric field @id from the columns (adjust to match your data)
# Example: Suppose your numeric field is '@id': 'numeric:diagnosis_interval_days'
# and a categorical/group field is '@id': 'text:sex'
# You may need to change these to match your schema.

numeric_field_id = None
group_field_id = None
df = dataframes[main_record_set]

# Try to automatically select
for c in df.columns:
    if df[c].dtype in ['int64', 'float64'] and numeric_field_id is None:
        numeric_field_id = c
    if (df[c].dtype == 'object' or str(df[c].dtype).startswith('category')) and group_field_id is None:
        group_field_id = c

if numeric_field_id:
    print(f"Numeric field selected: {numeric_field_id}")
else:
    print('No numeric field found. Please set `numeric_field_id` manually.')

if group_field_id:
    print(f"Group (categorical) field selected: {group_field_id}")
else:
    print('No categorical group field found. Please set `group_field_id` manually.')

# Set a threshold (adjust according to your field, here we use median as example)
if numeric_field_id:
    threshold = df[numeric_field_id].median()  # or choose a constant

    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nRecords where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a group field, if present
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.to_frame())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we'll plot the distribution of the selected numeric field (by `@id`) using matplotlib, and if group field available, a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=12)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, examine, and process a FAIR^2 dataset defined by a Croissant schema using the `mlcroissant` library. Record sets and fields were referenced by their Croissant `@id`, and sample exploratory analysis and visualizations were performed for key numeric and grouping fields.

This approach can be reused to explore other FAIR datasets defined with Croissant!